# Bronze Layer
Load raw CSVs from Volume into bronze schema tables.  
All columns loaded as STRING — Silver layer handles type casting.

## Setup Connection

In [1]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]
VOLUME = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Define Ingestion Configuration

In [2]:
from clickzetta.zettapark.types import StructType, StructField, StringType

def _str(*cols):
    return StructType([StructField(c, StringType()) for c in cols])

INGESTION_CONFIG = [
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_crm/cust_info.csv",     "table": "crm_cust_info",
     "schema": _str("cst_id","cst_key","cst_firstname","cst_lastname","cst_marital_status","cst_gndr","cst_create_date")},
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_crm/prd_info.csv",      "table": "crm_prd_info",
     "schema": _str("prd_id","prd_key","prd_nm","prd_cost","prd_line","prd_start_dt","prd_end_dt")},
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_crm/sales_details.csv", "table": "crm_sales_details",
     "schema": _str("sls_ord_num","sls_prd_key","sls_cust_id","sls_order_dt","sls_ship_dt","sls_due_dt","sls_sales","sls_quantity","sls_price")},
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_erp/CUST_AZ12.csv",     "table": "erp_cust_az12",
     "schema": _str("cid","bdate","gen")},
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_erp/LOC_A101.csv",      "table": "erp_loc_a101",
     "schema": _str("cid","cntry")},
    {"path": f"vol://{SCHEMA}.{VOLUME}/source_erp/PX_CAT_G1V2.csv",  "table": "erp_px_cat_g1v2",
     "schema": _str("id","cat","subcat","maintenance")},
]

## Ingest Files into Bronze Tables

In [3]:
for item in INGESTION_CONFIG:
    target = f"{SCHEMA}.{item['table']}"
    print(f"Ingesting → {target}")
    df = (
        session.read
               .option("header", "true")
               .schema(item["schema"])
               .csv(item["path"])
    )
    df.write.save_as_table(target, mode="overwrite")
    print("  OK")

Ingesting → public.crm_cust_info


  OK
Ingesting → public.crm_prd_info


  OK
Ingesting → public.crm_sales_details


  OK
Ingesting → public.erp_cust_az12


  OK
Ingesting → public.erp_loc_a101


  OK
Ingesting → public.erp_px_cat_g1v2


  OK


## Sanity Check

In [4]:
session.table(f"{SCHEMA}.crm_cust_info").limit(5).show()

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|
+------+----------+-------------+------------+------------------+--------+---------------+

